# Import the necessary libraries.

In [4]:
import pandas as pd
import numpy as np

<a id="section0"></a>
#  Phân Tích Dữ Liệu Tiểu Đường (Diabetes Analysis)

## Điều hướng nhanh
* [1. Tải & Tiền xử lý dữ liệu](#section1)
* [2. Tạo nhóm nguy cơ (Risk Group)](#section2)
* [3. Phân tích tương quan (Correlation)](#section3)
* [4. Lọc dữ liệu "Báo động đỏ"](#section4)
* [5. Báo cáo tổng hợp theo lứa tuổi](#section5)

---

<a id="section1"></a>
# 1. Tải & Tiền xử lý dữ liệu

Thực hiện truy vấn dữ liệu và xử lý các giá trị không hợp lệ (lỗi nhập liệu).

* **Vấn đề:** Các chỉ số như `BMI`, `BloodPressure` bằng $0$ là vô lý trong y tế.

* **Giải pháp xử lý:**
    1. Thay thế các giá trị $0$ thành `NaN` để không làm lệch kết quả thống kê.
    2. Điền (`Impute`) bằng giá trị **Trung vị (Median)** theo từng nhóm tuổi để đảm bảo tính khách quan.

[⬆ Quay lại mục lục](#section0)

---

In [8]:
df = pd.read_csv('diabetes.csv')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [9]:
# Đổi các giá trị từ 0 thành NaN
col_replace = ['Glucose', 'BloodPressure', 'BMI']
df[col_replace] = df[col_replace].replace(0, np.nan)

# Đổi giá trị thành median của nhóm tuổi đó
df['Age_Group'] = (df['Age'] // 10 ) * 10
for col in col_replace:
    df[col] = df.groupby('Age_Group')[col].transform(lambda x: x.fillna(x.median()))

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,Age_Group
0,6,148.0,72.0,35,0,33.6,0.627,50,1,50
1,1,85.0,66.0,29,0,26.6,0.351,31,0,30
2,8,183.0,64.0,0,0,23.3,0.672,32,1,30
3,1,89.0,66.0,23,94,28.1,0.167,21,0,20
4,0,137.0,40.0,35,168,43.1,2.288,33,1,30


<a id="section2"></a>

## 2. Phân loại BMI & Tương quan Glucose
Tạo nhóm nguy cơ và phân tích sự khác biệt về chỉ số đường huyết.

| Nhóm nguy cơ | Chỉ số BMI | Tình trạng |
| :--- | :--- | :--- |
| **Underweight** | $< 18.5$ | Thiếu cân |
| **Normal** | $18.5 \rightarrow 25$ | Bình thường |
| **Overweight** | $25 \rightarrow 30$ | Thừa cân |
| **Obese** | $> 30$ | Béo phì |

<div style="height: 20px;"></div>

> **Phân tích Correlaton:** Tính toán tỷ lệ chênh lệch (%) của chỉ số **Glucose** trung bình giữa nhóm `Obese` so với nhóm `Normal`.

[⬆ Quay lại mục lục](#section0)

---

In [10]:
# Tạo khoảng để chia
bins = [0, 18.5, 25, 30, float('inf')]
# Tạo nhãn cho nó
labels = ['Underweight', 'Normal', 'Overweight', 'Obese']

# Chia nhóm theo những gì mình tạo
df['BMI_Group'] = pd.cut(df['BMI'], bins = bins, labels = labels, right = True)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,Age_Group,BMI_Group
0,6,148.0,72.0,35,0,33.6,0.627,50,1,50,Obese
1,1,85.0,66.0,29,0,26.6,0.351,31,0,30,Overweight
2,8,183.0,64.0,0,0,23.3,0.672,32,1,30,Normal
3,1,89.0,66.0,23,94,28.1,0.167,21,0,20,Overweight
4,0,137.0,40.0,35,168,43.1,2.288,33,1,30,Obese


<a id="section3"></a>
## 3. Phân tích tương quan (Correlation)
Tập trung nghiên cứu mối quan hệ giữa thể trạng (BMI) và lượng đường trong máu (Glucose).

* **Câu hỏi nghiên cứu:** Chỉ số **Glucose trung bình** của nhóm `Obese` cao hơn nhóm `Normal` bao nhiêu %?

* **Công thức tính:** $$\Delta\% = \frac{\text{Mean Glucose}_{\text{Obese}} - \text{Mean Glucose}_{\text{Normal}}}{\text{Mean Glucose}_{\text{Normal}}} \times 100$$

<br>

[⬆ Quay lại mục lục](#section0)

---

In [12]:
# Tính trung bình của từng nhóm đối tượng
glucose_mean = df.groupby('BMI_Group')['Glucose'].mean()

# Tính xem để coi lệch bao nhiêu %
diff_mean = ((glucose_mean['Obese'] - glucose_mean['Normal']) / glucose_mean['Normal'] * 100).round(2)
print(f"Chỉ số trung bình của Obese cao hơn Normal là: {diff_mean}%")

Chỉ số trung bình của Obese cao hơn Normal là: 16.84%


<a id="section4"></a>

## 4. Lọc dữ liệu "Báo động đỏ" (Red Alert)

Trích xuất danh sách các đối tượng có nguy cơ cao nhất cần can thiệp y tế ngay lập tức:
* **Độ tuổi:** Trên 50 tuổi ($Age > 50$).
* **Thể trạng:** Thuộc nhóm `Obese`.
* **Kết quả:** Đã xác định bị tiểu đường (`Outcome = 1`).

[⬆ Quay lại mục lục](#section0)

---

In [13]:
# Tạo dataframe chứa các bệnh nhân có báo động đỏ
red_alert = df[(df['Age'] > 50) & (df['Outcome'] == 1) & (df['BMI_Group'] == 'Obese')]
print(f"Số người dính mức báo động đỏ là: {len(red_alert)}")

Số người dính mức báo động đỏ là: 30


<a id="section5"></a>

## 5. Báo cáo tổng hợp theo lứa tuổi
Thống kê tỷ lệ mắc bệnh để tìm ra xu hướng theo thời gian.

* **Phương pháp:** Chia nhóm tuổi theo thập kỷ (Decade Binning): `20s`, `30s`, `40s`,...

* **Chỉ số báo cáo:** Tỷ lệ (%) mắc tiểu đường trên tổng số người trong mỗi nhóm tuổi.

[⬆ Quay lại mục lục](#section0)

In [14]:
# Tính tỉ lệ trung bình của từng nhóm tuối bị tiểu đường
age_diabetes = round((df.groupby('Age_Group')['Outcome'].mean() * 100), 2)
 
# Sắp xếp lại theo độ tuổi
age_diabetes = age_diabetes.reset_index(name='diabetes_rate_pct')
#Sắp xếp theo thấp đến cao
#age_diabetes = age_diabetes.sort_values(ascending= False)
print("\nTỷ lệ % tiểu đường theo độ tuổi:")
print(age_diabetes)


Tỷ lệ % tiểu đường theo độ tuổi:
   Age_Group  diabetes_rate_pct
0         20              21.21
1         30              46.06
2         40              55.08
3         50              59.65
4         60              27.59
5         70              50.00
6         80               0.00
